# EMPCA training (frequency domain)

Step 1: imports + load traces + load PSD.

## Imports

In [5]:
import numpy as np
import h5py
import sys
from pathlib import Path

# Import optimized EMPCA implementation from reusable/EMPCA
_empca_candidates = [
    Path.cwd() / 'reusable' / 'EMPCA',
    Path.cwd().parent / 'reusable' / 'EMPCA',
    Path.cwd().parent.parent / 'reusable' / 'EMPCA',
]
_empca_dir = next((p for p in _empca_candidates if p.exists()), None)
if _empca_dir is None:
    raise FileNotFoundError('Could not locate reusable/EMPCA from current working directory')
sys.path.insert(0, str(_empca_dir))

from empca_TCY_optimized import EMPCA, ti_rfft

# ----------------------------
# 1) Baseline correction
# ----------------------------
def baseline_correct_per_trace(X_time, pretrigger=4000, method="mean"):
    """
    X_time: (n_events, n_time)
    pretrigger: number of first samples assumed pulse-free
    method: "mean" or "median" baseline estimate per trace
    Returns:
      X0: baseline-subtracted traces (same shape)
      baseline: (n_events,) baseline values
    """
    X_time = np.asarray(X_time, dtype=np.float64)
    if X_time.ndim != 2:
        raise ValueError(f"X_time must be 2D; got {X_time.shape}")
    if not (1 <= pretrigger <= X_time.shape[1]):
        raise ValueError("pretrigger must be within [1, n_time]")

    pre = X_time[:, :pretrigger]
    if method == "mean":
        baseline = np.mean(pre, axis=1)
    elif method == "median":
        baseline = np.median(pre, axis=1)
    else:
        raise ValueError("method must be 'mean' or 'median'")

    X0 = X_time - baseline[:, None]
    return X0, baseline


# ----------------------------
# 2) Frequency-domain shift-invariant transform
# ----------------------------
def to_shift_invariant_spectrum(X_time):
    """
    Applies the paper’s phase-difference transform implemented as ti_rfft.
    Output is complex with shape (n_events, n_freq).
    """
    X_tilde = ti_rfft(X_time)  # complex, shift-invariant frequency representation
    # ti_rfft may squeeze singleton batch dimensions; normalize to (n_events, n_freq)
    X_tilde = np.atleast_2d(X_tilde)
    if X_tilde.ndim != 2:
        raise RuntimeError(f"Unexpected output shape from ti_rfft: {X_tilde.shape}")
    return X_tilde


# ----------------------------
# 3) Build weights matrix from PSD
# ----------------------------
def make_inverse_psd_weights(noise_psd, eps=1e-18):
    """
    noise_psd: (n_freq,) one-sided PSD matching rfft bins.
    Returns W: (n_freq, n_freq) diagonal matrix with entries 1/(PSD+eps).
    """
    noise_psd = np.asarray(noise_psd, dtype=np.float64)
    if noise_psd.ndim != 1:
        raise ValueError(f"noise_psd must be 1D; got {noise_psd.shape}")
    inv = 1.0 / (noise_psd + eps)
    W = np.diag(inv)
    return W


# ----------------------------
# 4) Train EM-PCA
# ----------------------------
def train_empca_paper_style(
    X_time,
    noise_psd,
    n_comp=6,            # paper used 2 components for the estimator
    pretrigger=2000,
    baseline_method="mean",
    n_iter=50,
    mode="fast",         # fast corresponds to per-component approximation in the paper
    window=15, polyord=3, deriv=0,
):
    # 4.1 baseline subtraction
    X0, baseline = baseline_correct_per_trace(
        X_time, pretrigger=pretrigger, method=baseline_method
    )

    # 4.2 shift-invariant frequency representation
    X_tilde = to_shift_invariant_spectrum(X0)  # (n_events, n_freq) complex

    # 4.3 ensure PSD matches frequency dimension
    n_freq = X_tilde.shape[1]
    if len(noise_psd) != n_freq:
        raise ValueError(
            f"PSD length mismatch: PSD has {len(noise_psd)} bins but data has {n_freq}. "
            f"For n_time={X_time.shape[1]}, expected n_freq = n_time//2 + 1."
        )

    # 4.4 weights W = diag(1/PSD)
    W = make_inverse_psd_weights(noise_psd)

    # 4.5 EM-PCA fit (weighted, smoothed templates)
    pca = EMPCA(n_comp=n_comp)  # :contentReference[oaicite:6]{index=6}
    chi2s = pca.fit(
        X_tilde, W,
        n_iter=n_iter,
        mode=mode,
        window=window, polyord=polyord, deriv=deriv,
        verbose=False
    )

    return pca, chi2s, baseline, X_tilde, W


# ----------------------------
# 5) Pulse-height estimator: coefficient norm (paper)
# ----------------------------
def pca_amplitude_estimator(coeff):
    """
    coeff: (n_events, n_comp) complex or real coefficients
    Paper uses the (summed) norm of coefficients for n_comp=2:
      A_hat = sqrt(|b1|^2 + |b2|^2)
    Generalizes naturally to n_comp>2.
    """
    coeff = np.asarray(coeff)
    return np.sqrt(np.sum(np.abs(coeff)**2, axis=1))




## Load traces

In [6]:
# Load matched traces and rqs from HDF5
traces_h5 = "k_alpha_traces.h5"
rqs_h5 = "k_alpha_rqs.h5"

with h5py.File(traces_h5, "r") as f:
    X = f["traces"][:].astype(np.float64)

with h5py.File(rqs_h5, "r") as f:
    rqs = f["rqs"][:]

print("X shape:", X.shape)
print("rqs shape:", rqs.shape)


X shape: (4358, 32768)
rqs shape: (4358,)


## Load noise PSD

In [7]:
psd_path = "/ceph/dwong/delight/noise_psd_xray.npy"
psd_arr = np.load(psd_path)
psd = psd_arr[1] if psd_arr.ndim == 2 and psd_arr.shape[0] == 2 else psd_arr
psd = psd.astype(np.float64)

print("PSD shape:", psd.shape)


PSD shape: (16385,)


In [8]:
# ----------------------------
# Train two models on K_a traces
# 1) PSD_Ka_EMPCA: kernel = 1 / noise PSD
# 2) SNR2_Ka_EMPCA: kernel = |H_template|^2 / noise PSD
# ----------------------------
training_params = {
    "n_comp": 2,
    "pretrigger": 4000,
    "baseline_method": "mean",
    "n_iter": 30,
    "mode": "fast",
    "window": 15,
    "polyord": 3,
    "deriv": 0,
}

# Shared baseline + shift-invariant transform for fair comparison
X0, baselines = baseline_correct_per_trace(
    X,
    pretrigger=training_params["pretrigger"],
    method=training_params["baseline_method"],
)
X_tilde = to_shift_invariant_spectrum(X0)
n_freq = X_tilde.shape[1]

if len(psd) != n_freq:
    raise ValueError(
        f"PSD length mismatch: PSD has {len(psd)} bins but data has {n_freq}."
    )

# ----------------------------
# Model A: PSD_Ka_EMPCA
# ----------------------------
W_psd = make_inverse_psd_weights(psd)

pca_psd = EMPCA(n_comp=training_params["n_comp"])
chi2s_psd = pca_psd.fit(
    X_tilde,
    W_psd,
    n_iter=training_params["n_iter"],
    mode=training_params["mode"],
    window=training_params["window"],
    polyord=training_params["polyord"],
    deriv=training_params["deriv"],
    verbose=False,
)

# ----------------------------
# Model B: SNR2_Ka_EMPCA
# kernel = |H_template|^2 / PSD
# ----------------------------
transformed_template_candidates = [
    Path("/ceph/dwong/delight/transformed_template_K_alpha_tight.npy"),
    Path("transformed_template_K_alpha_tight.npy"),
    Path("wk7/PC_interpretation/transformed_template_K_alpha_tight.npy"),
]
transformed_template_path = next((q for q in transformed_template_candidates if q.exists()), None)

if transformed_template_path is not None:
    H = np.load(transformed_template_path)
    print("Using transformed template:", transformed_template_path)
else:
    raw_template_candidates = [
        Path("template_K_alpha_tight.npy"),
        Path("wk7/PC_interpretation/template_K_alpha_tight.npy"),
        Path("reusable/k_alpha/template_K_alpha_tight.npy"),
    ]
    raw_template_path = next((q for q in raw_template_candidates if q.exists()), None)
    if raw_template_path is None:
        raise FileNotFoundError(
            "Could not find transformed_template_K_alpha_tight.npy or template_K_alpha_tight.npy"
        )
    template_time = np.asarray(np.load(raw_template_path), dtype=np.float64).reshape(-1)
    if template_time.shape[0] != X.shape[1]:
        raise ValueError(
            f"Template length {template_time.shape[0]} does not match trace length {X.shape[1]}"
        )
    # Build a transformed template with the same representation as X_tilde.
    H = to_shift_invariant_spectrum(template_time[None, :])[0]
    print("Using raw template and transforming on the fly:", raw_template_path)

H = np.asarray(H)
if H.shape[0] != n_freq:
    raise ValueError(f"Transformed template length {H.shape[0]} != n_freq {n_freq}")

snr2_kernel = (np.abs(H) ** 2) / np.maximum(psd, 1e-18)
W_snr2 = np.diag(snr2_kernel)

pca_snr2 = EMPCA(n_comp=training_params["n_comp"])
chi2s_snr2 = pca_snr2.fit(
    X_tilde,
    W_snr2,
    n_iter=training_params["n_iter"],
    mode=training_params["mode"],
    window=training_params["window"],
    polyord=training_params["polyord"],
    deriv=training_params["deriv"],
    verbose=False,
)

# Save SMALL model artifacts (PCs only, no coefficients / no training trace matrices)
import pickle

PSD_Ka_EMPCA_path = "PSD_Ka_EMPCA.pkl"
PSD_Ka_EMPCA = {
    "model_name": "PSD_Ka_EMPCA",
    "n_comp": int(training_params["n_comp"]),
    "eigvec": np.asarray(pca_psd.eigvec),
    "training_params": training_params,
    "weight_type": "1/PSD",
    "psd_path": psd_path,
    "final_chi2": float(chi2s_psd[-1]),
    "n_iter_ran": int(len(chi2s_psd)),
    "n_events": int(X.shape[0]),
    "n_time": int(X.shape[1]),
    "n_freq": int(n_freq),
}
with open(PSD_Ka_EMPCA_path, "wb") as f:
    pickle.dump(PSD_Ka_EMPCA, f)

SNR2_Ka_EMPCA_path = "SNR2_Ka_EMPCA.pkl"
SNR2_Ka_EMPCA = {
    "model_name": "SNR2_Ka_EMPCA",
    "n_comp": int(training_params["n_comp"]),
    "eigvec": np.asarray(pca_snr2.eigvec),
    "training_params": training_params,
    "weight_type": "|H_template|^2/PSD",
    "psd_path": psd_path,
    "transformed_template_path": str(transformed_template_path) if transformed_template_path else None,
    "final_chi2": float(chi2s_snr2[-1]),
    "n_iter_ran": int(len(chi2s_snr2)),
    "n_events": int(X.shape[0]),
    "n_time": int(X.shape[1]),
    "n_freq": int(n_freq),
}
with open(SNR2_Ka_EMPCA_path, "wb") as f:
    pickle.dump(SNR2_Ka_EMPCA, f)

print("saved model to", PSD_Ka_EMPCA_path)
print("saved model to", SNR2_Ka_EMPCA_path)
print("final chi2 PSD_Ka_EMPCA:", PSD_Ka_EMPCA["final_chi2"])
print("final chi2 SNR2_Ka_EMPCA:", SNR2_Ka_EMPCA["final_chi2"])



 47%|████▋     | 14/30 [00:11<00:13,  1.21it/s]


Using raw template and transforming on the fly: template_K_alpha_tight.npy


 37%|███▋      | 11/30 [00:11<00:19,  1.00s/it]

saved model to PSD_Ka_EMPCA.pkl
saved model to SNR2_Ka_EMPCA.pkl
final chi2 PSD_Ka_EMPCA: 6.223297592748321e+18
final chi2 SNR2_Ka_EMPCA: 8.654137277683007e+25
